# 02 — Calidad y limpieza de datos



In [1]:
import pandas as pd
import numpy as np
import json

# Rutas relativas a la carpeta notebooks/
RUTA_RAW       = '../data/raw/streaming_users_dirty.json'
RUTA_PROCESSED = '../data/processed/streaming_users_clean.csv'
RUTA_LOG       = '../logs/pipeline_log.csv'

## Configurar la auditoría (preservar original + registrar)



In [3]:
with open(RUTA_RAW, encoding='utf-8') as f:
    data = json.load(f)

df_raw = pd.DataFrame(data)
n_inicial = len(df_raw)
df = df_raw.copy()

import os
if os.path.exists(RUTA_LOG):
    log = pd.read_csv(RUTA_LOG).to_dict('records')
    print(f"Log existente cargado desde 01_inspeccion_inicial.ipynb ({len(log)} paso(s) previos)")

def registrar(df, paso, desc):
    log.append({
        'Paso': paso,
        'Descripción': desc,
        'Filas': len(df),
        'Nulos': int(df.isnull().sum().sum()),
        'Retención (%)': round(len(df) / n_inicial * 100, 2)
    })

if not log:
    registrar(df, 0, "Dataset original")

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

Log existente cargado desde 01_inspeccion_inicial.ipynb (1 paso(s) previos)
Filas: 8160 | Columnas: 8


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


## Paso 1 — Duplicados



In [4]:
dup_id = df['user_id'].duplicated().sum()
dup_filas = df.duplicated().sum()
print(f"user_id duplicados : {dup_id}")
print(f"Filas 100% idénticas: {dup_filas}")

n_antes = len(df)
df = df.drop_duplicates(subset='user_id').reset_index(drop=True)
print(f"\nFilas eliminadas: {n_antes - len(df)}")

registrar(df, 1, "Eliminación de duplicados (user_id)")

user_id duplicados : 160
Filas 100% idénticas: 126

Filas eliminadas: 160


## Paso 2 — Normalización de categóricas



In [5]:
print("Antes:")
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f"  {col}: {sorted(df[col].dropna().unique())[:6]} ...")

Antes:
  subscription_plan: ['BASICO', 'Basic', 'Básico', 'Estándar', 'Estándar ', 'PREMIUM'] ...
  country: ['ARG', 'Argentina', 'Argentina ', 'BRA', 'Brasil', 'Brazil'] ...
  favorite_genre: ['ACCIÓN', 'Acción', 'Action', 'COMEDIA', 'CRIME', 'Comedia'] ...


In [6]:
mapa_plan = {
    'básico':'Básico', 'basico':'Básico', 'basic':'Básico',
    'estándar':'Estándar', 'std':'Estándar', 'estandar':'Estándar', 'standard':'Estándar',
    'premium':'Premium', 'premiun':'Premium',
}

mapa_country = {
    'argentina':'Argentina', 'arg':'Argentina',
    'brasil':'Brasil', 'brazil':'Brasil', 'bra':'Brasil',
    'chile':'Chile', 'chl':'Chile',
    'colombia':'Colombia', 'col':'Colombia',
    'méxico':'México', 'mexico':'México', 'mex':'México',
    'perú':'Perú', 'peru':'Perú', 'per':'Perú',
    'uruguay':'Uruguay', 'ury':'Uruguay',
}

mapa_genre = {
    'acción':'Acción', 'accion':'Acción', 'action':'Acción',
    'comedia':'Comedia', 'comedy':'Comedia',
    'crime':'Crime', 'crimen':'Crime',
    'documental':'Documental', 'documentary':'Documental', 'doc':'Documental',
    'drama':'Drama',
    'romance':'Romance',
    'thriller':'Thriller', 'thriler':'Thriller',
}

nulos_antes = df[['subscription_plan', 'country', 'favorite_genre']].isnull().sum().sum()

df['subscription_plan'] = df['subscription_plan'].str.strip().str.lower().map(mapa_plan)
df['country']           = df['country'].str.strip().str.lower().map(mapa_country)
df['favorite_genre']    = df['favorite_genre'].str.strip().str.lower().map(mapa_genre)

nulos_despues = df[['subscription_plan', 'country', 'favorite_genre']].isnull().sum().sum()
print(f"Nulos en las 3 columnas -> antes de mapear: {nulos_antes} | después de mapear: {nulos_despues}")
print("(si 'después' > 'antes', hay valores que el diccionario no contempla: revisar value_counts())")

registrar(df, 2, "Normalización de subscription_plan, country, favorite_genre")

Nulos en las 3 columnas -> antes de mapear: 240 | después de mapear: 240
(si 'después' > 'antes', hay valores que el diccionario no contempla: revisar value_counts())


In [7]:
print("Después:")
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f"  {col}: {sorted(df[col].dropna().unique())}")

Después:
  subscription_plan: ['Básico', 'Estándar', 'Premium']
  country: ['Argentina', 'Brasil', 'Chile', 'Colombia', 'México', 'Perú', 'Uruguay']
  favorite_genre: ['Acción', 'Comedia', 'Crime', 'Documental', 'Drama', 'Romance', 'Thriller']


## Paso 3 — Valores imposibles → NaN



In [8]:
# --- age ---
age_invalido = df[(df['age'] < 0) | (df['age'] > 100)]
print(f"age inválidos: {len(age_invalido)}")
df.loc[(df['age'] < 0) | (df['age'] > 100), 'age'] = np.nan

# --- monthly_watch_time_mins ---
watch_invalido = df[df['monthly_watch_time_mins'] < 0]
print(f"monthly_watch_time_mins negativos: {len(watch_invalido)}")
df.loc[df['monthly_watch_time_mins'] < 0, 'monthly_watch_time_mins'] = np.nan

# --- customer_support_tickets ---
tickets_invalido = df[df['customer_support_tickets'] < 0]
print(f"customer_support_tickets negativos: {len(tickets_invalido)}")
df.loc[df['customer_support_tickets'] < 0, 'customer_support_tickets'] = np.nan

registrar(df, 3, "Corrección de valores imposibles (age, monthly_watch_time_mins, customer_support_tickets)")

age inválidos: 74
monthly_watch_time_mins negativos: 49
customer_support_tickets negativos: 29


## Paso 4 — Tratamiento de outliers



In [9]:
# --- age: revisar outliers, sin evidencia de error -> se conserva ---
Q1, Q3 = df['age'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lim_sup = Q3 + 1.5 * IQR
out_age = df[df['age'] > lim_sup]
print(f"age — outliers k=1.5: {len(out_age)} ({len(out_age)/df['age'].notna().sum()*100:.2f}%), máximo real: {df['age'].max()}")
print("Decisión: CONSERVAR (cola natural, valores plausibles, sin patrón de error)")

age — outliers k=1.5: 17 (0.21%), máximo real: 80.0
Decisión: CONSERVAR (cola natural, valores plausibles, sin patrón de error)


In [10]:
# --- monthly_watch_time_mins: detectar y tratar valores imposibles ---
imposibles_watch = df['monthly_watch_time_mins'].isin([99999.0, 50000.0])
print(f"Valores imposibles detectados (99999 / 50000): {imposibles_watch.sum()}")

df.loc[imposibles_watch, 'monthly_watch_time_mins'] = np.nan

# Revisar la cola alta que queda, ya sin los valores imposibles
Q1, Q3 = df['monthly_watch_time_mins'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lim_sup_30 = Q3 + 3.0 * IQR
out_resto = df[df['monthly_watch_time_mins'] > lim_sup_30]
print(f"Outliers restantes (k=3.0, sin valores imposibles): {len(out_resto)}, máximo: {df['monthly_watch_time_mins'].max():.1f}")
print("Decisión valores imposibles: tratar como inválidos -> NaN (no son mediciones reales)")
print("Decisión cola restante: CONSERVAR -> decae de forma continua, compatible con uso real alto")

Valores imposibles detectados (99999 / 50000): 31
Outliers restantes (k=3.0, sin valores imposibles): 108, máximo: 4193.7
Decisión valores imposibles: tratar como inválidos -> NaN (no son mediciones reales)
Decisión cola restante: CONSERVAR -> decae de forma continua, compatible con uso real alto


In [11]:
# --- customer_support_tickets: detectar y tratar valores imposibles ---
print("Distribución completa antes del tratamiento:")
print(df['customer_support_tickets'].value_counts().sort_index())

imposibles_tickets = df['customer_support_tickets'].isin([99.0, 150.0])
print(f"\nValores imposibles detectados (99 / 150): {imposibles_tickets.sum()}")

df.loc[imposibles_tickets, 'customer_support_tickets'] = np.nan
print("Decisión valores imposibles: tratar como inválidos -> NaN")
print("Decisión resto (0-5): CONSERVAR -> distribución de conteo natural y decreciente")

registrar(df, 4, "Tratamiento de outliers: valores imposibles en monthly_watch_time_mins y customer_support_tickets")

Distribución completa antes del tratamiento:
customer_support_tickets
0.0      3563
1.0      2819
2.0      1157
3.0       280
4.0        71
5.0        14
99.0       35
150.0      32
Name: count, dtype: int64

Valores imposibles detectados (99 / 150): 67
Decisión valores imposibles: tratar como inválidos -> NaN
Decisión resto (0-5): CONSERVAR -> distribución de conteo natural y decreciente


## Paso 5 — Diagnóstico del mecanismo de falta (MCAR / MAR / MNAR)


In [12]:
columnas_con_nulos = ['age', 'monthly_watch_time_mins', 'favorite_genre', 'last_login_date', 'customer_support_tickets']

for col in columnas_con_nulos:
    falta = df[col].isnull()
    print(f"--- {col} ({falta.sum()} faltantes, {falta.mean()*100:.2f}%) ---")
    for grupo in ['subscription_plan', 'country']:
        tasa = df.groupby(grupo)[col].apply(lambda x: x.isnull().mean() * 100).round(2).sort_values(ascending=False)
        print(f"  Tasa de falta por {grupo}:")
        print(tasa.to_string())
    print()

--- age (74 faltantes, 0.92%) ---
  Tasa de falta por subscription_plan:
subscription_plan
Básico      1.00
Estándar    0.99
Premium     0.63
  Tasa de falta por country:
country
Chile        1.20
Uruguay      0.96
Brasil       0.95
México       0.95
Argentina    0.90
Colombia     0.88
Perú         0.62

--- monthly_watch_time_mins (273 faltantes, 3.41%) ---
  Tasa de falta por subscription_plan:
subscription_plan
Premium     10.74
Estándar     2.27
Básico       1.08
  Tasa de falta por country:
country
Uruguay      4.29
Brasil       3.55
México       3.37
Perú         3.35
Chile        3.35
Argentina    3.08
Colombia     2.89

--- favorite_genre (240 faltantes, 3.00%) ---
  Tasa de falta por subscription_plan:
subscription_plan
Básico      3.14
Estándar    3.02
Premium     2.65
  Tasa de falta por country:
country
Brasil       3.29
México       3.29
Chile        3.18
Perú         2.91
Colombia     2.89
Argentina    2.71
Uruguay      2.71

--- last_login_date (320 faltantes, 4.00%) ---

## Paso 6 — Imputación diferenciada por mecanismo



In [13]:
# age (MCAR) -> mediana global
df['age'] = df['age'].fillna(df['age'].median())

# monthly_watch_time_mins (MAR por subscription_plan) -> mediana por grupo
df['monthly_watch_time_mins'] = df.groupby('subscription_plan')['monthly_watch_time_mins'].transform(
    lambda x: x.fillna(x.median())
)

# favorite_genre (MCAR, categórica) -> moda global
moda_genre = df['favorite_genre'].mode()[0]
df['favorite_genre'] = df['favorite_genre'].fillna(moda_genre)

# customer_support_tickets (MCAR) -> mediana global
df['customer_support_tickets'] = df['customer_support_tickets'].fillna(df['customer_support_tickets'].median())

# last_login_date (MCAR) -> NO se imputa, ver justificación en la celda de abajo
print(f"Nulos en last_login_date (sin imputar, a propósito): {df['last_login_date'].isnull().sum()}")

registrar(df, 5, "Imputación diferenciada: age/tickets (mediana global), watch_time (mediana por plan-MAR), genre (moda) — last_login_date NO se imputa")

print(f"\nNulos totales post-imputación: {df.isnull().sum().sum()}")
print(df.isnull().sum()[df.isnull().sum() > 0])

Nulos en last_login_date (sin imputar, a propósito): 320

Nulos totales post-imputación: 320
last_login_date    320
dtype: int64


## Guardar resultado final + log


In [14]:
df.to_csv(RUTA_PROCESSED, index=False)

log_df = pd.DataFrame(log)
log_df.to_csv(RUTA_LOG, index=False)

print(log_df.to_string(index=False))
print(f"\nRetención estructural actual: {log_df['Retención (%)'].iloc[-1]}%")
print(f"\nDataset procesado guardado en: {RUTA_PROCESSED}")
print(f"Log guardado en: {RUTA_LOG}")

 Paso                                                                                                                           Descripción  Filas  Nulos  Retención (%)
    0                                                                     Dataset original (carga e inspección inicial, sin modificaciones)   8160    753         100.00
    1                                                                                                   Eliminación de duplicados (user_id)   8000    753          98.04
    2                                                                           Normalización de subscription_plan, country, favorite_genre   8000    753          98.04
    3                                             Corrección de valores imposibles (age, monthly_watch_time_mins, customer_support_tickets)   8000    905          98.04
    4                                     Tratamiento de outliers: valores imposibles en monthly_watch_time_mins y customer_support_tickets   8000   1003  